In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.flightpulse import drift
from src.flightpulse.features import build_features
from src.flightpulse.queries import load_window

ref = build_features(load_window("2019-06-01", "2019-06-30"))
cur = build_features(load_window("2020-04-01", "2020-04-30"))  # COVID window

res = drift.evidently_drift(ref, cur, html_name="ref2019_vs_covid2020.html")
print("dataset_drift:", res["dataset_drift"])
print("drifted_share:", res["drifted_share"])
print("drifted_count:", res["drifted_count"])

d:\Projects\FlightPulse\src\flightpulse\queries.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(start_date, end_date),
d:\Projects\FlightPulse\src\flightpulse\queries.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(start_date, end_date),


dataset_drift: True
drifted_share: 0.6666666666666666
drifted_count: 8.0


In [ ]:
ctrl = drift.null_control(ref)
print("NULL CONTROL:")
print(drift.psi_report(ref, ctrl), "\n")   

dd = drift.shift_distance(ref, factor=1.3)
print("INJECTED DISTANCE DRIFT:")
print(drift.psi_report(ref, dd), "\n") 

ld = drift.inject_label_drift(ref, target_rate=0.35)
print("INJECTED LABEL DRIFT:")
print("ref positive rate:", round(ref['label'].mean(), 3),
      "-> injected:", round(ld['label'].mean(), 3), "\n")

cat = drift.resample_airline_mix(ref, boost="WN", frac=1.0)
res_cat = drift.evidently_drift(ref, cat, html_name="airline_mix_drift.html")
print("INJECTED AIRLINE-MIX DRIFT (Evidently):",
      "dataset_drift =", res_cat["dataset_drift"])

NULL CONTROL:
             column     psi level
0          distance  0.0001  none
1  crs_elapsed_time  0.0001  none
2       dep_minutes  0.0001  none
3        congestion  0.0001  none 

INJECTED DISTANCE DRIFT:
             column     psi     level
0          distance  0.1315  moderate
1  crs_elapsed_time  0.0000      none
2       dep_minutes  0.0000      none
3        congestion  0.0000      none 

INJECTED LABEL DRIFT:
ref positive rate: 0.239 -> injected: 0.35 

INJECTED AIRLINE-MIX DRIFT (Evidently): dataset_drift = False


In [ ]:
res_dc = drift.deepchecks_drift(ref, cur)
if res_dc["available"]:
    res_dc["feature_drift"].show()   
    res_dc["label_drift"].show()
else:
    print("Deepchecks not installed — skipping (Evidently + PSI/KS cover drift).")

Deepchecks not installed — skipping (Evidently + PSI/KS cover drift).
